# SCEX-CPDP: Low-Memory Research Notebook
## Stability-aware Cost-sensitive Explainable Cross-Project Defect Prediction

**Designed for Google Colab with limited RAM.** This version avoids loading all 77 CSV files into one DataFrame. It scans headers first, lets you select only the intended PROMISE datasets, loads one file at a time, keeps numeric software metrics, downcasts data types, and limits rows/features if needed.

### Important research safeguards
- Do not treat all 77 CSVs as independent projects. They may include duplicates, versions, or unrelated files.
- Check the printed file list and project mapping before running experiments.
- The notebook uses a conservative, reproducible baseline experiment. It does not fabricate results.
- NASA MDP datasets have different schemas and should be analyzed separately unless a justified harmonization protocol is established.
- LOC-based effort is only computed when a valid LOC feature is present; it is a proxy, not developer effort ground truth.


## 1. Runtime setup

In [ ]:
# Run this cell after Runtime -> Restart session.
!pip -q install scikit-learn scipy matplotlib seaborn

In [ ]:
import os, re, gc, json, warnings, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, balanced_accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)

## 2. Download or locate PROMISE-backup

In [ ]:
REPO_DIR = Path("/content/PROMISE-backup")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/feiwww/PROMISE-backup.git", str(REPO_DIR)],
        check=True
    )

CSV_FILES = sorted(REPO_DIR.rglob("*.csv"))
print("CSV files discovered:", len(CSV_FILES))
assert CSV_FILES, "No CSV files found. Check the repository download." 

## 3. Inspect CSV headers only (low RAM)

This cell reads **zero data rows** from each file. It reports possible defect target columns and file sizes. It does not concatenate files.

In [ ]:
TARGET_ALIASES = {
    "bug", "bugs", "defect", "defects", "number_of_bugs",
    "bug_count", "bugcount", "numberofbugs"
}

def find_target_column(columns):
    for col in columns:
        normalized = re.sub(r"[^a-z0-9]+", "", str(col).lower())
        if normalized in {re.sub(r"[^a-z0-9]+", "", x) for x in TARGET_ALIASES}:
            return col
    return None

FILE_INFO = []
for idx, path in enumerate(CSV_FILES):
    try:
        header = pd.read_csv(path, nrows=0).columns.tolist()
        target = find_target_column(header)
        FILE_INFO.append({
            "index": idx,
            "file": str(path),
            "relative_file": str(path.relative_to(REPO_DIR)),
            "size_mb": round(path.stat().st_size / (1024**2), 3),
            "target": target,
            "columns": header
        })
    except Exception as exc:
        print("Header read failed:", path.name, str(exc)[:120])

info_df = pd.DataFrame([
    {k: v for k, v in x.items() if k != "columns"} for x in FILE_INFO
])
display(info_df.head(100))
print("Files with a recognized defect target:",
      int(info_df["target"].notna().sum()) if len(info_df) else 0)

## 4. Select only the datasets you intend to study

**Do not automatically select every file.** Inspect the file names above. Set `SELECTED_FILE_INDICES` to the indices of the intended Jureczko/PROMISE CSV files. You can select a small number first to test the pipeline.

Example: `SELECTED_FILE_INDICES = [4, 7, 12]`

If you leave it as `None`, the notebook selects at most the first 8 target-bearing files as a *smoke test only*, not as a final research dataset.

In [ ]:
# EDIT THIS after inspecting info_df.
SELECTED_FILE_INDICES = None  # Example: [4, 7, 12]

# Memory safety controls
MAX_ROWS_PER_FILE = 20000      # Set None for all rows after confirming RAM is sufficient
MAX_FEATURES = 40              # Keep at most this many numeric features
MAX_FILE_SIZE_MB = 100         # Skip unusually large files by default; increase only if needed
MIN_ROWS_PER_PROJECT = 20

if SELECTED_FILE_INDICES is None:
    available = [x["index"] for x in FILE_INFO
                 if x["target"] is not None and x["size_mb"] <= MAX_FILE_SIZE_MB]
    SELECTED_FILE_INDICES = available[:8]
    print("SMOKE TEST selection (first up to 8 files):", SELECTED_FILE_INDICES)
    print("For final experiments, explicitly select and justify the files.")

selected_info = [x for x in FILE_INFO
                 if x["index"] in SELECTED_FILE_INDICES and x["target"] is not None]
print("Selected files:", len(selected_info))
for x in selected_info:
    print(x["index"], x["relative_file"], "|", x["size_mb"], "MB", "| target:", x["target"])

if not selected_info:
    raise ValueError("No selected files with recognized defect target. Review section 3 and target aliases.")

## 5. Project-name mapping (must be audited)

The project name is inferred from the file path. This is only a starting point. Review and correct `PROJECT_OVERRIDES` after inspecting the filenames. If one file represents a release/version, keep its project identity consistent with the other releases of that project.

In [ ]:
KNOWN_PROJECTS = [
    "ant", "camel", "ivy", "jedit", "log4j", "lucene",
    "poi", "synapse", "velocity", "xalan", "xerces"
]

def infer_project(path_string):
    stem = Path(path_string).stem.lower()
    full = str(path_string).lower()
    for project in KNOWN_PROJECTS:
        if re.search(rf"(^|[^a-z0-9]){re.escape(project)}([^a-z0-9]|$)", full):
            return project
    # fallback: use filename stem, remove common version-like suffixes
    stem = re.sub(r"[-_ ]?(v|version)?\d+(\.\d+)*$", "", stem)
    return re.sub(r"[^a-z0-9]+", "_", stem).strip("_") or "unknown"

PROJECT_OVERRIDES = {
    # Example:
    # "ant-1.7.csv": "ant",
}

for x in selected_info:
    fname = Path(x["file"]).name
    x["project"] = PROJECT_OVERRIDES.get(fname, infer_project(x["file"]))
    print(fname, "=>", x["project"])

## 6. Low-memory loader

- Reads one selected file at a time.
- Reads only numeric columns plus the defect target.
- Optionally samples a maximum number of rows per file.
- Converts numeric columns to `float32` and labels to `int8`.
- Drops identifier-like columns and constant columns.
- Does **not** keep the original full CSVs in memory.

For a first run, retain the defaults. For final experiments, report any row cap or sampling strategy in the paper.

In [ ]:
ID_HINTS = ("name", "file", "class", "id", "version", "module", "project")

def load_one_dataset(meta):
    path = meta["file"]
    target_col = meta["target"]

    # Header already inspected; select numeric columns without reading all columns.
    header = pd.read_csv(path, nrows=0).columns.tolist()
    numeric_probe = pd.read_csv(path, nrows=100)
    numeric_cols = numeric_probe.select_dtypes(include=[np.number]).columns.tolist()
    del numeric_probe
    gc.collect()

    # Always retain target, even if pandas initially inferred it as object.
    usecols = list(dict.fromkeys(numeric_cols + [target_col]))
    if target_col not in usecols:
        usecols.append(target_col)

    df = pd.read_csv(path, usecols=lambda c: c in usecols, low_memory=True)

    if MAX_ROWS_PER_FILE is not None and len(df) > MAX_ROWS_PER_FILE:
        # Deterministic sampling without replacement
        df = df.sample(n=MAX_ROWS_PER_FILE, random_state=SEED)

    # Convert target to numeric; rows without a usable target are removed.
    y_numeric = pd.to_numeric(df[target_col], errors="coerce")
    valid = y_numeric.notna()
    df = df.loc[valid].copy()
    y_numeric = y_numeric.loc[valid]

    # Defect prediction is binary: any bug count > 0 means defective.
    y = (y_numeric > 0).astype("int8")

    X = df.drop(columns=[target_col], errors="ignore")
    X = X.select_dtypes(include=[np.number])

    # Drop likely identifiers and constant columns.
    drop_cols = [
        c for c in X.columns
        if any(h in str(c).lower() for h in ID_HINTS)
    ]
    X = X.drop(columns=drop_cols, errors="ignore")
    X = X.replace([np.inf, -np.inf], np.nan)

    # Float32 reduces memory usage by roughly half versus float64.
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")

    nunique = X.nunique(dropna=True)
    X = X.loc[:, nunique > 1]

    # Keep bounded number of features, deterministically by column order.
    if MAX_FEATURES is not None and X.shape[1] > MAX_FEATURES:
        X = X.iloc[:, :MAX_FEATURES]

    result = X.reset_index(drop=True)
    result["__label__"] = y.reset_index(drop=True).astype("int8")
    result["__project__"] = meta["project"]
    result["__source_file__"] = Path(path).name

    del df, X, y, y_numeric
    gc.collect()
    return result

project_frames = {}
for j, meta in enumerate(selected_info, 1):
    print(f"Loading {j}/{len(selected_info)}: {Path(meta['file']).name}")
    one = load_one_dataset(meta)
    project = meta["project"]
    project_frames.setdefault(project, []).append(one)
    print("  rows:", len(one), "| numeric features:", one.shape[1] - 3,
          "| defective rate:", round(float(one["__label__"].mean()), 4))
    del one
    gc.collect()

# Combine only selected files, grouped by project.
data_by_project = {}
for project, parts in project_frames.items():
    combined = pd.concat(parts, ignore_index=True, sort=False)
    data_by_project[project] = combined
    print(project, combined.shape, "RAM MB:", round(combined.memory_usage(deep=True).sum()/1024**2, 2))
    del parts
gc.collect()

print("Projects loaded:", list(data_by_project.keys()))

## 7. Harmonize features conservatively

Cross-project models require a common feature space. This notebook uses the **intersection** of numeric feature names across selected projects. This avoids fabricating mappings between differently named metrics.

If the intersection is too small, stop and define a defensible feature-mapping protocol rather than silently using unrelated columns.

In [ ]:
projects = sorted(data_by_project.keys())
if len(projects) < 2:
    raise ValueError("At least two distinct projects are needed for cross-project evaluation.")

feature_sets = [
    set(c for c in data_by_project[p].columns
        if c not in {"__label__", "__project__", "__source_file__"})
    for p in projects
]
COMMON_FEATURES = sorted(set.intersection(*feature_sets))
print("Projects:", projects)
print("Common feature count:", len(COMMON_FEATURES))
print("Common features:", COMMON_FEATURES)

if len(COMMON_FEATURES) < 2:
    raise ValueError(
        "Too few common features. Inspect schemas and create a justified feature mapping."
    )

# Restrict each project's frame to the same common feature space.
for p in projects:
    data_by_project[p] = data_by_project[p][COMMON_FEATURES + ["__label__"]].copy()
    gc.collect()

## 8. Define models and evaluation metrics

Evaluation is leave-one-project-out (LOPO): train on all selected projects except one, test on the held-out project. The imputer/scaler are fitted inside each model pipeline using training data only.

Cost-sensitive model: Random Forest with class-balanced weights. This is a baseline cost-sensitive treatment; it does not claim a calibrated business-cost optimum. Set the cost ratio only after defining it in the study.

In [ ]:
MODELS = {
    "Dummy_majority": DummyClassifier(strategy="most_frequent"),
    "Logistic_Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced",
                                     random_state=SEED))
    ]),
    "Random_Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=120, max_depth=12, min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=1,
            random_state=SEED
        ))
    ]),
    "Extra_Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=120, max_depth=12, min_samples_leaf=2,
            class_weight="balanced", n_jobs=1,
            random_state=SEED
        ))
    ])
}

def evaluate_predictions(y_true, y_pred, y_score):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }
    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
        out["pr_auc"] = average_precision_score(y_true, y_score)
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan
    return out

## 9. Run LOPO cross-project experiments

In [ ]:
results = []
predictions = []

for test_project in projects:
    train_projects = [p for p in projects if p != test_project]

    train_parts = [data_by_project[p] for p in train_projects]
    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = data_by_project[test_project]

    X_train = train_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
    y_train = train_df["__label__"].to_numpy(dtype=np.int8, copy=True)
    X_test = test_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
    y_test = test_df["__label__"].to_numpy(dtype=np.int8, copy=True)

    print(f"\nHeld-out: {test_project} | train={len(y_train)} test={len(y_test)}")

    for model_name, model_template in MODELS.items():
        # Fresh model instance for every fold
        import sklearn.base
        model = sklearn.base.clone(model_template)
        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            raw_scores = model.decision_function(X_test)
            scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-9)
        else:
            scores = np.zeros(len(y_test), dtype=float)

        pred = (scores >= 0.5).astype("int8")
        metrics = evaluate_predictions(y_test, pred, scores)
        results.append({
            "test_project": test_project,
            "model": model_name,
            "train_projects": ",".join(train_projects),
            "n_train": len(y_train),
            "n_test": len(y_test),
            **metrics
        })

        predictions.append(pd.DataFrame({
            "test_project": test_project,
            "model": model_name,
            "y_true": y_test,
            "score": scores,
            "prediction": pred
        }))

        del model
        gc.collect()

    del train_parts, train_df, X_train, y_train, X_test, y_test
    gc.collect()

results_df = pd.DataFrame(results)
display(results_df)
print("Completed LOPO folds:", results_df["test_project"].nunique())

## 10. Summarize results and save outputs

Report per-project results as well as macro averages. Do not report only pooled averages, because project difficulty can vary substantially.

In [ ]:
summary_df = (
    results_df.groupby("model")
    [["accuracy", "precision", "recall", "f1", "mcc",
      "balanced_accuracy", "roc_auc", "pr_auc"]]
    .mean()
    .sort_values("f1", ascending=False)
)
display(summary_df)

OUT_DIR = Path("/content/SCEX_CPDP_results")
OUT_DIR.mkdir(exist_ok=True)

results_df.to_csv(OUT_DIR / "lopo_per_project_results.csv", index=False)
summary_df.to_csv(OUT_DIR / "model_macro_summary.csv")
pd.concat(predictions, ignore_index=True).to_csv(
    OUT_DIR / "lopo_predictions.csv", index=False
)

with open(OUT_DIR / "experiment_metadata.json", "w") as f:
    json.dump({
        "seed": SEED,
        "selected_files": [x["relative_file"] for x in selected_info],
        "project_mapping": {x["relative_file"]: x["project"] for x in selected_info},
        "projects": projects,
        "common_features": COMMON_FEATURES,
        "max_rows_per_file": MAX_ROWS_PER_FILE,
        "max_features": MAX_FEATURES,
        "target_definition": "defective = bug_count > 0",
        "validation": "leave-one-project-out"
    }, f, indent=2)

print("Saved result files to:", OUT_DIR)

## 11. Visualize model performance

In [ ]:
plot_df = summary_df.reset_index()
plt.figure(figsize=(9, 4))
plt.bar(plot_df["model"], plot_df["f1"])
plt.ylabel("Macro F1")
plt.title("LOPO cross-project defect prediction")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(plot_df["model"], plot_df["mcc"])
plt.ylabel("Macro MCC")
plt.title("LOPO cross-project MCC")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 12. Basic stability analysis across random seeds (optional)

This section reruns Random Forest with several seeds for each held-out project. It reports mean and standard deviation of F1. It is intentionally limited to one model to control runtime and memory.

In [ ]:
RUN_SEED_STABILITY = True
STABILITY_SEEDS = [11, 22, 33]  # Add more seeds for final study if compute permits.

stability_rows = []
if RUN_SEED_STABILITY:
    for test_project in projects:
        train_df = pd.concat(
            [data_by_project[p] for p in projects if p != test_project],
            ignore_index=True
        )
        test_df = data_by_project[test_project]

        Xtr = train_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
        ytr = train_df["__label__"].to_numpy(dtype=np.int8, copy=True)
        Xte = test_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
        yte = test_df["__label__"].to_numpy(dtype=np.int8, copy=True)

        for seed in STABILITY_SEEDS:
            clf = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", RandomForestClassifier(
                    n_estimators=100, max_depth=12, min_samples_leaf=2,
                    class_weight="balanced_subsample", n_jobs=1,
                    random_state=seed
                ))
            ])
            clf.fit(Xtr, ytr)
            yp = clf.predict(Xte)
            stability_rows.append({
                "test_project": test_project,
                "seed": seed,
                "f1": f1_score(yte, yp, zero_division=0),
                "mcc": matthews_corrcoef(yte, yp)
            })
            del clf
            gc.collect()

        del train_df, Xtr, ytr, Xte, yte
        gc.collect()

stability_df = pd.DataFrame(stability_rows)
if not stability_df.empty:
    display(stability_df.groupby("test_project")[["f1", "mcc"]].agg(["mean", "std"]))
    stability_df.to_csv(OUT_DIR / "seed_stability.csv", index=False)

## 13. Feature importance and explanation (optional, low-memory)

This uses permutation importance on one held-out fold and one fitted Random Forest. It avoids installing SHAP and avoids storing large explanation arrays. Importance is predictive association, **not causality**.

In [ ]:
RUN_PERMUTATION_IMPORTANCE = True

if RUN_PERMUTATION_IMPORTANCE and len(projects) >= 2:
    test_project = projects[-1]
    train_df = pd.concat(
        [data_by_project[p] for p in projects if p != test_project],
        ignore_index=True
    )
    test_df = data_by_project[test_project]

    Xtr = train_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
    ytr = train_df["__label__"].to_numpy(dtype=np.int8, copy=True)
    Xte = test_df[COMMON_FEATURES].to_numpy(dtype=np.float32, copy=True)
    yte = test_df["__label__"].to_numpy(dtype=np.int8, copy=True)

    explain_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=100, max_depth=12, min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=1, random_state=SEED
        ))
    ])
    explain_model.fit(Xtr, ytr)

    # Manual permutation importance using F1; subsample test rows for RAM/time.
    rng = np.random.default_rng(SEED)
    n_eval = min(2000, len(yte))
    idx = rng.choice(len(yte), size=n_eval, replace=False)
    X_eval = Xte[idx].copy()
    y_eval = yte[idx].copy()
    base_pred = explain_model.predict(X_eval)
    base_f1 = f1_score(y_eval, base_pred, zero_division=0)

    importance = []
    for j, feature in enumerate(COMMON_FEATURES):
        X_perm = X_eval.copy()
        X_perm[:, j] = rng.permutation(X_perm[:, j])
        perm_pred = explain_model.predict(X_perm)
        perm_f1 = f1_score(y_eval, perm_pred, zero_division=0)
        importance.append((feature, base_f1 - perm_f1))
        del X_perm
    importance_df = pd.DataFrame(
        importance, columns=["feature", "f1_decrease_after_permutation"]
    ).sort_values("f1_decrease_after_permutation", ascending=False)

    display(importance_df.head(20))
    importance_df.to_csv(OUT_DIR / "permutation_feature_importance.csv", index=False)

    del train_df, test_df, Xtr, ytr, Xte, yte, explain_model
    gc.collect()

## 14. Statistical comparison (exploratory)

Wilcoxon signed-rank tests compare paired per-project F1 values for two models. With few projects, statistical power is low; treat p-values as exploratory and report the number of paired projects.

In [ ]:
wide_f1 = results_df.pivot(index="test_project", columns="model", values="f1")
model_names = list(wide_f1.columns)

if len(model_names) >= 2:
    comparison_rows = []
    reference = model_names[0]
    for other in model_names[1:]:
        pair = wide_f1[[reference, other]].dropna()
        if len(pair) >= 2 and not np.allclose(pair[reference], pair[other]):
            stat, pval = wilcoxon(pair[reference], pair[other])
        else:
            stat, pval = np.nan, np.nan
        comparison_rows.append({
            "model_1": reference, "model_2": other,
            "n_projects": len(pair), "wilcoxon_stat": stat, "p_value": pval
        })
    display(pd.DataFrame(comparison_rows))

## 15. Optional LOC effort proxy (only if a LOC feature exists)

This estimates recall among the highest-ranked modules within a LOC budget. It is a **proxy** based on module LOC, not measured inspection effort. It is omitted when no LOC column is available.

In [ ]:
loc_candidates = [
    c for c in COMMON_FEATURES
    if c.lower() in {"loc", "locom", "loc_total", "total_loc", "loc_code"}
]
print("LOC candidate columns:", loc_candidates)

def recall_at_loc_budget(y_true, scores, loc, budget_fraction=0.20):
    loc = np.asarray(loc, dtype=float)
    loc = np.where(np.isfinite(loc) & (loc > 0), loc, 1.0)
    order = np.argsort(-scores)
    budget = budget_fraction * loc.sum()
    cum_loc = np.cumsum(loc[order])
    selected = order[cum_loc <= budget]
    if len(selected) == 0:
        selected = order[:1]
    total_defects = int(np.sum(y_true))
    if total_defects == 0:
        return np.nan
    return float(np.sum(np.asarray(y_true)[selected]) / total_defects)

if loc_candidates:
    loc_col = loc_candidates[0]
    # Use predictions from the Random Forest LOPO folds
    rf_preds = pd.concat(
        [p for p in predictions if p["model"].iloc[0] == "Random_Forest"],
        ignore_index=True
    )
    # Match test-project LOC values in the same row order used by predictions.
    effort_rows = []
    for p in projects:
        if p not in data_by_project:
            continue
        pred_p = rf_preds[rf_preds["test_project"] == p]
        loc_vals = data_by_project[p][loc_col].fillna(
            data_by_project[p][loc_col].median()
        ).to_numpy()
        if len(pred_p) == len(loc_vals):
            effort_rows.append({
                "test_project": p,
                "recall_at_20pct_loc": recall_at_loc_budget(
                    pred_p["y_true"].to_numpy(),
                    pred_p["score"].to_numpy(),
                    loc_vals
                )
            })
    effort_df = pd.DataFrame(effort_rows)
    display(effort_df)
    effort_df.to_csv(OUT_DIR / "loc_effort_proxy.csv", index=False)
else:
    print("No recognized LOC feature. Effort proxy skipped.")

## 16. Download experiment outputs

The ZIP contains per-project metrics, macro summary, predictions, metadata, and optional explanation/stability outputs.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/SCEX_CPDP_results", "zip", OUT_DIR)
print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Outside Colab: find the ZIP in", zip_path)

## 17. Research checklist before writing the paper

1. Replace smoke-test selection with a documented, manually audited dataset selection.
2. Verify every project identity and release/version mapping.
3. Verify the defect target semantics for each dataset; confirm whether values are bug counts or binary labels.
4. Document excluded files, duplicate handling, row caps, missingness, and class imbalance.
5. Report LOPO per-project results and macro averages; do not present the smoke-test results as final.
6. Add appropriate baselines and ablations before claiming novelty or superiority.
7. Define the false-negative/false-positive cost ratio from a defensible software-maintenance rationale. Class weighting alone is not a complete cost model.
8. Add repeated seeds and confidence intervals where computationally feasible.
9. Treat feature importance as explanation of model behavior, not causal evidence.
10. Keep NASA MDP experiments separate unless feature harmonization is explicitly justified.

**Memory note:** If Colab still runs out of RAM, lower `MAX_ROWS_PER_FILE` (e.g. 5000), lower `MAX_FEATURES` (e.g. 20), and select fewer projects. Keep `n_jobs=1`; do not load all 77 files at once.
